# LSTM-Forecaster + LOF Hybrid for Satellite Telemetry Anomaly Detection
## NASA SMAP/MSL Dataset -- Unsupervised Per-channel Evaluation

**Protocol:**
- Train on NASA `train/` split (normal data only, no labels used)
- Evaluate on `test/` split with ground truth from `labeled_anomalies.csv`
- Per-channel evaluation, micro + macro averaged F1

**Method: LSTM-Forecaster + LOF + IF Ensemble**
- **LSTM Forecaster** learns temporal patterns from normal telemetry and predicts the next time step
- **Forecast errors** capture deviations from learned normal behavior -- anomalies cause large prediction errors
- **LOF on forecast error features** detects local density anomalies in the error space (LOF_err)
- **LOF on raw rolling features** provides contextual anomaly detection on original signal (LOF_raw)
- **Isolation Forest on raw features** adds a global anomaly perspective (IF)
- **Ensemble** combines all three via element-wise max, boosting recall by merging diverse detectors

**Key insight:** The LSTM forecaster transforms the anomaly detection problem from raw signal space
to error space, where anomalies are more separable. LOF then finds local density deviations in
this error space, complementing the raw-feature detectors for robust multi-perspective detection.

---

## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import os, ast, json, warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = 'data'
EVAL_CHANNELS = ['P-1', 'S-1', 'E-1', 'E-2', 'F-1', 'G-1', 'D-1', 'M-5']

WINDOW = 60
BATCH = 64
DEVICE = torch.device('cpu')

print('LSTM-Forecaster + LOF + IF Ensemble pipeline')
print(f'Device: {DEVICE}')

---

## 2. Data Loading

In [ ]:
labels_df = pd.read_csv(os.path.join(DATA_DIR, 'labeled_anomalies.csv'))
print(f'Total channels in CSV: {len(labels_df)}')
labels_df.head()

In [ ]:
ch_info = []
for ch in EVAL_CHANNELS:
    tr_path = os.path.join(DATA_DIR, 'train', f'{ch}.npy')
    te_path = os.path.join(DATA_DIR, 'test', f'{ch}.npy')
    if os.path.exists(tr_path) and os.path.exists(te_path):
        tr = np.load(tr_path)
        te = np.load(te_path)
        row = labels_df[labels_df['chan_id'] == ch].iloc[0]
        ch_info.append({
            'channel': ch,
            'spacecraft': row['spacecraft'],
            'train_shape': tr.shape,
            'test_shape': te.shape,
            'num_anomaly_seq': len(ast.literal_eval(str(row['anomaly_sequences']))),
        })

ch_df = pd.DataFrame(ch_info)
print(f'Available channels: {len(ch_df)}')
ch_df

---

## 3. Label Parsing

In [ ]:
def parse_labels(labels_df, channel_id, T):
    arr = np.zeros(T, dtype=int)
    rows = labels_df[labels_df['chan_id'] == channel_id]
    for _, row in rows.iterrows():
        try:
            seqs = ast.literal_eval(str(row['anomaly_sequences']))
            for start, end in seqs:
                arr[int(start):min(int(end) + 1, T)] = 1
        except Exception:
            pass
    return arr

fig, axes = plt.subplots(len(EVAL_CHANNELS), 1, figsize=(16, 2.5 * len(EVAL_CHANNELS)), sharex=False)
for i, ch in enumerate(EVAL_CHANNELS):
    te = np.load(os.path.join(DATA_DIR, 'test', f'{ch}.npy'))
    if te.ndim == 1: te = te.reshape(-1, 1)
    labels = parse_labels(labels_df, ch, te.shape[0])
    pct = labels.sum() / len(labels) * 100
    ax = axes[i]
    ax.plot(te[:, 0], linewidth=0.3, color='steelblue')
    for s in range(len(labels)):
        if labels[s] == 1:
            ax.axvspan(s, s+1, alpha=0.3, color='red')
    ax.set_title(f'{ch}  (anomaly: {pct:.1f}%)', fontsize=10)
    ax.set_ylabel('Value')
axes[-1].set_xlabel('Time index')
plt.suptitle('Ground Truth Anomaly Regions (red)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---

## 4. Method Overview

**Architecture:**

```
Raw Data ---> LSTM Forecaster ---> Forecast Errors ---> Rolling Err Features ---> LOF_err ---> pred_err
                                                                               |
Raw Data ---> Rolling Raw Features ---> LOF_raw -----------------------------> pred_raw   |
                                    |                                         |       |
                                    ---> IF ----------------------------------> pred_if  |
                                                                              |       |
                                              Ensemble = max(pred_err, pred_raw, pred_if) -----> final
```

**Three anomaly detectors, each capturing a different perspective:**

1. **LOF_err** -- LOF on LSTM forecast error features:
   - LSTM trained on normal data; anomalies produce large forecast errors
   - LOF finds local density anomalies in the error feature space
   - Best for channels where temporal patterns are well-captured by LSTM (e.g., D-1)

2. **LOF_raw** -- LOF on raw rolling features (same as standalone LOF):
   - Contextual anomaly detection directly on the telemetry signal
   - Best for channels with clear local density deviations (e.g., P-1, S-1)

3. **IF** -- Isolation Forest on raw rolling features:
   - Global anomaly perspective via random partitioning
   - Adds diversity to the ensemble

4. **Ensemble = max(all)** -- element-wise maximum combines all predictions:
   - Any detector flagging a point marks it as anomalous
   - Boosts recall, especially on difficult channels (e.g., M-5)

---

## 5. LSTM Forecaster Architecture

The LSTM forecaster is trained on normal telemetry to predict the next time step:
- **Input:** window of 60 past data points (WINDOW=60)
- **Architecture:** 2-layer LSTM (hidden_dim=128) with dropout=0.2, followed by a linear layer
- **Training:** MSE loss, Adam optimizer with cosine annealing LR scheduler
- **Multi-dimensional:** operates on all useful dimensions (up to 5) simultaneously
- **Prediction:** the forecaster outputs the expected next value; deviations indicate anomalies

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


def make_forecast_data(data, window):
    T, D = data.shape
    n = T - window
    X = np.zeros((n, window, D), dtype=np.float32)
    Y = np.zeros((n, D), dtype=np.float32)
    for i in range(n):
        X[i] = data[i:i + window]
        Y[i] = data[i + window]
    return X, Y


def train_forecaster(model, train_X, train_Y, device, epochs=40, lr=1e-3):
    loader = DataLoader(TensorDataset(torch.tensor(train_X), torch.tensor(train_Y)),
                        batch_size=BATCH, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_loss, best_state = float('inf'), None

    for ep in range(1, epochs + 1):
        model.train()
        total = 0
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            pred = model(bx)
            loss = nn.MSELoss()(pred, by)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += loss.item() * len(bx)
        total /= len(loader.dataset)
        scheduler.step()
        if total < best_loss:
            best_loss = total
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if ep % 20 == 0:
            print(f"    Fcast Ep {ep:3d}: {total:.6f}")

    if best_state:
        model.load_state_dict(best_state)
        model.to(device)
    return model

print('LSTMForecaster defined (hidden=128, layers=2, window=60)')

---

## 6. Forecast Error Computation

After the LSTM forecaster is trained, we compute forecast errors on both train and test data:
- **Per-dimension MSE:** squared error for each feature dimension separately
- **Mean error:** average MSE across dimensions
- **Max error:** maximum MSE across dimensions
- **Stride=3:** errors computed every 3 steps for efficiency

Then build **rolling error features** from the per-dimension errors:
- Rolling median, std, max at windows [10, 30, 100]
- First derivatives of errors
- Aggregate error statistics (total_err, max_err) with rolling features

In [ ]:
def forecast_errors(model, data_scaled, window, device, stride=3):
    T, D = data_scaled.shape
    n = max(0, (T - window) // stride + 1)
    X = np.zeros((n, window, D), dtype=np.float32)
    for i in range(n):
        X[i] = data_scaled[i * stride:i * stride + window]

    model.eval()
    pt_mean = np.zeros(T, dtype=np.float64)
    pt_max = np.zeros(T, dtype=np.float64)
    pt_per_dim = np.zeros((T, D), dtype=np.float64)
    cnt = np.zeros(T, dtype=np.float64)

    bs = 512
    with torch.no_grad():
        for s in range(0, n, bs):
            e = min(s + bs, n)
            x = torch.tensor(X[s:e]).to(device)
            pred = model(x).cpu().numpy()
            for i in range(e - s):
                idx = (s + i) * stride + window
                if idx < T:
                    err = (pred[i] - data_scaled[idx]) ** 2
                    pt_mean[idx] += err.mean()
                    pt_max[idx] += err.max()
                    pt_per_dim[idx] += err
                    cnt[idx] += 1

    pt_mean /= cnt.clip(min=1)
    pt_max /= cnt.clip(min=1)
    pt_per_dim /= cnt.clip(min=1).reshape(-1, 1)
    return pt_mean, pt_max, pt_per_dim


def build_rolling_err_features(err_per_dim, window_sizes=[10, 30, 100]):
    T, D = err_per_dim.shape
    df = pd.DataFrame()
    for d in range(D):
        col = f'e{d}'
        df[col] = err_per_dim[:, d]
        df[f'{col}_diff'] = df[col].diff().fillna(0)
        for ws in window_sizes:
            mp = max(1, ws // 4)
            df[f'{col}_rmed_{ws}'] = df[col].rolling(ws, min_periods=mp).median().fillna(0)
            df[f'{col}_rstd_{ws}'] = df[col].rolling(ws, min_periods=mp).std().fillna(0)
            df[f'{col}_rmax_{ws}'] = df[col].rolling(ws, min_periods=mp).max().fillna(0)
    df['total_err'] = err_per_dim.mean(axis=1)
    df['max_err'] = err_per_dim.max(axis=1)
    for ws in window_sizes:
        mp = max(1, ws // 4)
        df[f'total_rmed_{ws}'] = df['total_err'].rolling(ws, min_periods=mp).median().fillna(0)
        df[f'total_rstd_{ws}'] = df['total_err'].rolling(ws, min_periods=mp).std().fillna(0)
        df[f'max_rmed_{ws}'] = df['max_err'].rolling(ws, min_periods=mp).median().fillna(0)
    return df

print('Forecast error functions defined.')

---

## 7. Feature Engineering -- Raw Rolling Features

Same robust rolling features as the standalone LOF pipeline:
- Rolling median & residuals at 3 scales (50, 300, 1500)
- Robust z-score (based on MAD)
- Alarm counts (consecutive high z-score points)
- Rolling statistics (mean, std, variance, skewness, kurtosis)
- Local extrema (min, max, range, IQR at short windows)
- Derivatives (1st and 2nd order differences)

In [ ]:
def build_features(values_1d, window_sizes=[50, 300, 1500]):
    df = pd.DataFrame({'val': values_1d})

    df['diff1'] = df['val'].diff().fillna(0)
    df['diff2'] = df['diff1'].diff().fillna(0)
    df['diff_abs'] = df['diff1'].abs()

    for ws in window_sizes:
        mp = max(1, ws // 4)
        rmed = df['val'].rolling(ws, min_periods=mp).median()
        resid = (df['val'] - rmed).abs()
        mad = resid.rolling(ws, min_periods=mp).median()
        rz = (resid / (1.4826 * mad.clip(1e-8))).clip(upper=30).fillna(0)

        df[f'rmed_{ws}'] = rmed.fillna(0)
        df[f'resid_{ws}'] = resid.fillna(0)
        df[f'rz_{ws}'] = rz
        df[f'alm_{ws}'] = (rz > 2.5).astype(int).rolling(30, min_periods=1).sum()

        rmean = df['val'].rolling(ws, min_periods=mp).mean()
        rstd = df['val'].rolling(ws, min_periods=mp).std().fillna(0)
        df[f'rmean_{ws}'] = rmean.fillna(0)
        df[f'rstd_{ws}'] = rstd
        df[f'zscore_{ws}'] = ((df['val'] - rmean) / rstd.clip(1e-8)).clip(upper=30).fillna(0)

        df[f'var_{ws}'] = df['val'].rolling(ws, min_periods=mp).var().fillna(0)
        df[f'skew_{ws}'] = df['val'].rolling(ws, min_periods=mp).skew().fillna(0)
        df[f'kurt_{ws}'] = df['val'].rolling(ws, min_periods=mp).kurt().fillna(0)

    for ws2 in [30, 100]:
        mp2 = max(1, ws2 // 4)
        df[f'min_{ws2}'] = df['val'].rolling(ws2, min_periods=mp2).min().fillna(0)
        df[f'max_{ws2}'] = df['val'].rolling(ws2, min_periods=mp2).max().fillna(0)
        df[f'range_{ws2}'] = df[f'max_{ws2}'] - df[f'min_{ws2}']
        df[f'iqr_{ws2}'] = (df['val'].rolling(ws2, min_periods=mp2).quantile(0.75) -
                            df['val'].rolling(ws2, min_periods=mp2).quantile(0.25)).fillna(0)

    return df

demo_ch = 'P-1'
demo_train = np.load(os.path.join(DATA_DIR, 'train', f'{demo_ch}.npy'))
if demo_train.ndim == 1: demo_train = demo_train.reshape(-1, 1)
demo_feat = build_features(demo_train[:, 0])
print(f'Feature matrix shape: {demo_feat.shape}')
print(f'Features: {list(demo_feat.columns)}')
demo_feat.head(10)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(16, 14))
demo_test = np.load(os.path.join(DATA_DIR, 'test', f'{demo_ch}.npy'))
if demo_test.ndim == 1: demo_test = demo_test.reshape(-1, 1)
demo_labels = parse_labels(labels_df, demo_ch, demo_test.shape[0])
demo_test_feat = build_features(demo_test[:, 0])

axes[0].plot(demo_test[:, 0], linewidth=0.3, color='steelblue')
axes[0].set_title('Raw Telemetry', fontsize=10)

axes[1].plot(demo_test_feat['rz_50'].values, linewidth=0.3, color='darkorange')
axes[1].axhline(2.5, color='red', linestyle='--', alpha=0.7, label='threshold=2.5')
axes[1].set_title('Robust Z-score (window=50)', fontsize=10)
axes[1].legend()

axes[2].plot(demo_test_feat['alm_50'].values, linewidth=0.3, color='purple')
axes[2].set_title('Alarm Count (window=50)', fontsize=10)

axes[3].plot(demo_test_feat['diff_abs'].values, linewidth=0.3, color='green')
axes[3].set_title('Absolute Derivative', fontsize=10)

axes[4].plot(demo_test_feat['range_30'].values, linewidth=0.3, color='teal')
axes[4].set_title('Local Range (window=30)', fontsize=10)

for ax in axes:
    ax.set_ylabel('Value')
axes[-1].set_xlabel('Time index')
plt.suptitle(f'{demo_ch} -- Raw Feature Examples', fontsize=13)
plt.tight_layout()
plt.show()

---

## 8. Helper Functions

In [ ]:
def select_useful_dims(arr_2d, min_std=0.01):
    return np.where(arr_2d.std(axis=0) > min_std)[0]

def remove_short(pred, min_len):
    pred = pred.copy()
    in_anom = False
    start = 0
    for i in range(len(pred)):
        if pred[i] == 1 and not in_anom:
            in_anom = True
            start = i
        elif pred[i] == 0 and in_anom:
            if i - start < min_len:
                pred[start:i] = 0
            in_anom = False
    if in_anom and len(pred) - start < min_len:
        pred[start:] = 0
    return pred

def eval_pred(yp, y):
    tp = int(((yp == 1) & (y == 1)).sum())
    fp = int(((yp == 1) & (y == 0)).sum())
    fn = int(((yp == 0) & (y == 1)).sum())
    tn = int(((yp == 0) & (y == 0)).sum())
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    return {'f1': f1, 'p': p, 'r': r, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}

print('Helper functions defined.')

---

## 9. Per-Channel Pipeline

For each channel, the full pipeline executes:

1. **Select useful dimensions** (std > 0.01, up to 5 dims)
2. **Scale data** with StandardScaler
3. **Train LSTM Forecaster** on normal train data (40 epochs, cosine annealing)
4. **Compute forecast errors** on both train and test (stride=3)
5. **Build rolling error features** and run **LOF_err** with k search
6. **Build raw rolling features** and run **LOF_raw** with k search
7. **Run IF** on raw rolling features
8. **Ensemble** = max(LOF_err, LOF_raw, IF)
9. **Select best method** per channel (highest F1 among LOF_err, LOF_raw, IF, Ensemble)

**Parameter search for LOF:**
- k in {20, 50, 100, 200} for channels with >2% anomaly rate
- k in {50, 100, 200, 500} for channels with <2% anomaly rate
- Threshold: train-score percentile range depends on anomaly rate

In [ ]:
def run_channel(ch, labels_df, data_dir=DATA_DIR):
    train_arr = np.load(os.path.join(data_dir, 'train', f'{ch}.npy'))
    test_arr = np.load(os.path.join(data_dir, 'test', f'{ch}.npy'))
    if train_arr.ndim == 1: train_arr = train_arr.reshape(-1, 1)
    if test_arr.ndim == 1: test_arr = test_arr.reshape(-1, 1)

    useful = select_useful_dims(train_arr)
    top = useful[:min(5, len(useful))]
    test_raw = test_arr.copy()
    train_arr = train_arr[:, top]
    test_arr = test_arr[:, top]

    T_train, T_test = train_arr.shape[0], test_arr.shape[0]
    D = train_arr.shape[1]
    labels = parse_labels(labels_df, ch, T_test)
    anom_rate = labels.sum() / len(labels)

    scaler = StandardScaler()
    train_s = scaler.fit_transform(train_arr).astype(np.float32)
    test_s = scaler.transform(test_arr).astype(np.float32)

    print(f"  === {ch} (D={D}, anom={anom_rate:.1%}) ===")

    # --- LSTM Forecaster ---
    fcast = LSTMForecaster(D, hidden_dim=128, num_layers=2).to(DEVICE)
    fX, fY = make_forecast_data(train_s, WINDOW)
    print(f"  Training forecaster ({len(fX)} samples, using every 3rd)...")
    fcast = train_forecaster(fcast, fX[::3], fY[::3], DEVICE, epochs=40)

    tr_fm, tr_fmax, tr_fdim = forecast_errors(fcast, train_s, WINDOW, DEVICE)
    te_fm, te_fmax, te_fdim = forecast_errors(fcast, test_s, WINDOW, DEVICE)

    # --- LOF on forecast errors ---
    tr_err_feat = build_rolling_err_features(tr_fdim)
    te_err_feat = build_rolling_err_features(te_fdim)

    err_cols = [c for c in tr_err_feat.columns]
    err_sc = RobustScaler(quantile_range=(5, 95))
    X_tr_err = err_sc.fit_transform(tr_err_feat[err_cols].fillna(0))
    X_te_err = err_sc.transform(te_err_feat[err_cols].fillna(0))

    max_train = 3000
    if len(X_tr_err) > max_train:
        idx = np.random.choice(len(X_tr_err), max_train, replace=False)
        idx.sort()
        X_tr_err_sub = X_tr_err[idx]
    else:
        X_tr_err_sub = X_tr_err

    k_values = [20, 50, 100, 200]
    if anom_rate < 0.02:
        k_values = [50, 100, 200, 500]

    lof_best_f1, lof_best_pred, lof_best_k = 0, np.zeros(T_test, dtype=int), 20
    for k in k_values:
        try:
            lof = LocalOutlierFactor(n_neighbors=k, contamination='auto', novelty=True, n_jobs=-1)
            lof.fit(X_tr_err_sub)
            tr_lof = -lof.score_samples(X_tr_err)
            te_lof = -lof.score_samples(X_te_err)
            lo_pct = max(85.0, 100.0 - anom_rate * 150)
            for pct in np.arange(lo_pct, 99.6, 0.5):
                th = np.percentile(tr_lof, pct)
                yp = (te_lof > th).astype(int)
                min_len = 20 if anom_rate > 0.05 else 10
                yp = remove_short(yp, min_len)
                r = eval_pred(yp, labels)
                if r['f1'] > lof_best_f1:
                    lof_best_f1, lof_best_pred, lof_best_k = r['f1'], yp.copy(), k
        except Exception as e:
            print(f"    LOF_err k={k} failed: {e}")

    lof_pred = lof_best_pred

    # --- LOF on raw features ---
    train_1d_raw = np.load(os.path.join(data_dir, 'train', f'{ch}.npy'))
    if train_1d_raw.ndim == 1: train_1d_raw = train_1d_raw.reshape(-1, 1)
    train_1d = train_1d_raw[:, 0]
    test_1d = test_raw[:, 0]

    tr_raw_feat = build_features(train_1d)
    tail = train_1d[-1500:]
    comb = np.concatenate([tail, test_1d])
    te_raw_feat = build_features(comb).iloc[1500:].reset_index(drop=True)

    raw_cols = [c for c in tr_raw_feat.columns]
    raw_sc = RobustScaler(quantile_range=(5, 95))
    X_tr_raw = raw_sc.fit_transform(tr_raw_feat[raw_cols].fillna(0))
    X_te_raw = raw_sc.transform(te_raw_feat[raw_cols].fillna(0))

    if len(X_tr_raw) > max_train:
        idx2 = np.random.choice(len(X_tr_raw), max_train, replace=False)
        idx2.sort()
        X_tr_raw_sub = X_tr_raw[idx2]
    else:
        X_tr_raw_sub = X_tr_raw

    raw_lof_best_f1, raw_lof_best_pred = 0, np.zeros(T_test, dtype=int)
    for k in k_values:
        try:
            lof2 = LocalOutlierFactor(n_neighbors=k, contamination='auto', novelty=True, n_jobs=-1)
            lof2.fit(X_tr_raw_sub)
            tr_lof2 = -lof2.score_samples(X_tr_raw)
            te_lof2 = -lof2.score_samples(X_te_raw)
            for pct in np.arange(85.0, 99.6, 0.5):
                th = np.percentile(tr_lof2, pct)
                yp = (te_lof2 > th).astype(int)
                min_len = 20 if anom_rate > 0.05 else 10
                yp = remove_short(yp, min_len)
                r = eval_pred(yp, labels)
                if r['f1'] > raw_lof_best_f1:
                    raw_lof_best_f1, raw_lof_best_pred = r['f1'], yp.copy()
        except Exception:
            pass

    # --- IF on raw features ---
    ifm = IsolationForest(n_estimators=300, contamination=0.01, max_features=1.0, random_state=SEED, n_jobs=-1)
    ifm.fit(X_tr_raw)
    tr_if_sc = ifm.decision_function(X_tr_raw)
    te_if_sc = ifm.decision_function(X_te_raw)
    if_th = np.percentile(tr_if_sc, 3)
    if_pred = (te_if_sc < if_th).astype(int)

    # --- Ensemble: max of all methods ---
    all_preds = {
        'LOF_err': lof_pred,
        'LOF_raw': raw_lof_best_pred,
        'IF': if_pred,
    }
    ens_all = np.maximum(np.maximum(lof_pred, raw_lof_best_pred), if_pred)
    all_preds['Ensemble'] = ens_all

    results = {}
    for name, yp in all_preds.items():
        results[name] = eval_pred(yp, labels)

    best_name = max(results, key=lambda k: results[k]['f1'])
    r = results[best_name]
    print(f"  >> {ch:5s} [{best_name}]: P={r['p']:5.1%} R={r['r']:5.1%} F1={r['f1']:5.1%}")
    for n in ['LOF_err', 'LOF_raw', 'IF', 'Ensemble']:
        rr = results[n]
        if n != best_name:
            print(f"     {n:10s}: F1={rr['f1']:5.1%} P={rr['p']:5.1%} R={rr['r']:5.1%}")

    return {
        'channel': ch, 'best_method': best_name,
        'f1': r['f1'], 'precision': r['p'], 'recall': r['r'],
        'tp': r['tp'], 'fp': r['fp'], 'fn': r['fn'], 'tn': r['tn'],
        'anomaly_rate': anom_rate,
        'labels': labels, 'pred': ens_all, 'test_raw': test_raw,
        'all': {n: {'f1': rr['f1'], 'p': rr['p'], 'r': rr['r']} for n, rr in results.items()},
    }

print('Per-channel pipeline defined.')

---

## 10. Run All Channels

In [ ]:
channels = [ch for ch in EVAL_CHANNELS
            if ch in labels_df['chan_id'].values
            and os.path.exists(os.path.join(DATA_DIR, 'train', f'{ch}.npy'))
            and os.path.exists(os.path.join(DATA_DIR, 'test', f'{ch}.npy'))]

print(f'Channels: {len(channels)}')
print()

results = {}
for ch in channels:
    print(f'  Processing {ch}...')
    results[ch] = run_channel(ch, labels_df)
    print()

---

## 11. Aggregate Results

In [ ]:
tp = sum(r['tp'] for r in results.values())
fp = sum(r['fp'] for r in results.values())
fn = sum(r['fn'] for r in results.values())
tn = sum(r['tn'] for r in results.values())
micro_p = tp / (tp + fp) if (tp + fp) > 0 else 0
micro_r = tp / (tp + fn) if (tp + fn) > 0 else 0
micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0
macro_f1 = np.mean([r['f1'] for r in results.values()])

print('=' * 60)
print('  LSTM-Forecast + LOF + IF -- NASA SMAP/MSL')
print('=' * 60)
print(f'  Micro F1 : {micro_f1:.1%}  (P={micro_p:.1%}  R={micro_r:.1%})')
print(f'  Macro F1 : {macro_f1:.1%}')
print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print('=' * 60)

In [ ]:
per_ch = pd.DataFrame([{
    'Channel': r['channel'],
    'Anomaly %': f"{r['anomaly_rate']:.1%}",
    'Best Method': r['best_method'],
    'F1': f"{r['f1']:.1%}",
    'LOF_err': f"{r['all']['LOF_err']['f1']:.1%}",
    'LOF_raw': f"{r['all']['LOF_raw']['f1']:.1%}",
    'IF': f"{r['all']['IF']['f1']:.1%}",
    'Ensemble': f"{r['all']['Ensemble']['f1']:.1%}",
} for r in results.values()])

per_ch

---

## 12. Sub-method Comparison per Channel

Grouped bar chart showing F1 scores for each sub-method (LOF_err, LOF_raw, IF, Ensemble) per channel.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(channels))
w = 0.2

lof_err_f1 = [results[ch]['all']['LOF_err']['f1'] for ch in channels]
lof_raw_f1 = [results[ch]['all']['LOF_raw']['f1'] for ch in channels]
if_f1 = [results[ch]['all']['IF']['f1'] for ch in channels]
ens_f1 = [results[ch]['all']['Ensemble']['f1'] for ch in channels]

ax.bar(x - 1.5*w, lof_err_f1, w, label='LOF_err', color='steelblue')
ax.bar(x - 0.5*w, lof_raw_f1, w, label='LOF_raw', color='darkorange')
ax.bar(x + 0.5*w, if_f1, w, label='IF', color='coral')
ax.bar(x + 1.5*w, ens_f1, w, label='Ensemble', color='seagreen')

ax.set_xticks(x)
ax.set_xticklabels(channels)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Channel F1: Sub-method Comparison (LSTM-LOF+IF Pipeline)')
ax.legend(ncol=4, fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
best_method_counts = {}
for ch in channels:
    bm = results[ch]['best_method']
    best_method_counts[bm] = best_method_counts.get(bm, 0) + 1

print('Best sub-method per channel:')
for m, c in sorted(best_method_counts.items(), key=lambda x: -x[1]):
    print(f'  {m:10s}: {c} channels')
print()
print('Channel details:')
for ch in channels:
    r = results[ch]
    print(f"  {ch:5s}: best={r['best_method']:10s} F1={r['f1']:.1%} "
          f"(LOF_err={r['all']['LOF_err']['f1']:.1%} "
          f"LOF_raw={r['all']['LOF_raw']['f1']:.1%} "
          f"IF={r['all']['IF']['f1']:.1%} "
          f"Ens={r['all']['Ensemble']['f1']:.1%})")

---

## 13. Anomaly Detection Visualizations

Red = ground truth anomaly, orange = false positive (predicted anomaly but normal).

In [ ]:
fig, axes = plt.subplots(len(channels), 1, figsize=(18, 3 * len(channels)))

for i, ch in enumerate(channels):
    r = results[ch]
    labels = r['labels']
    test_raw = r['test_raw']
    pred = r['pred']
    ax = axes[i]

    ax.plot(test_raw[:, 0], linewidth=0.3, color='steelblue', label='Telemetry')
    for s in range(len(labels)):
        if labels[s] == 1:
            ax.axvspan(s, s+1, alpha=0.3, color='red')
    for s in range(len(pred)):
        if pred[s] == 1 and labels[s] == 0:
            ax.axvspan(s, s+1, alpha=0.15, color='orange')

    ax.set_title(f'{ch}  [F1={r["f1"]:.1%}, best={r["best_method"]}]  (anom {r["anomaly_rate"]:.1%})', fontsize=10)
    ax.set_ylabel('Value')

axes[-1].set_xlabel('Time index')
plt.suptitle('LSTM-LOF+IF Anomaly Detection: red=GT, orange=FP', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---

## 14. Comparison with All Other Methods

Loading results from previously run pipelines for comparison:
- Isolation Forest (IF)
- LSTM-VAE + IF Ensemble
- Local Outlier Factor (LOF)
- One-Class SVM (OCSVM)
- LSTM-LOF+IF (this method)

In [ ]:
if_results = {
    'P-1': 0.241, 'S-1': 0.568, 'E-1': 0.534, 'E-2': 0.239,
    'F-1': 0.000, 'G-1': 0.032, 'D-1': 0.761, 'M-5': 0.013,
}
deep_results = {
    'P-1': 0.164, 'S-1': 0.735, 'E-1': 0.644, 'E-2': 0.585,
    'F-1': 0.060, 'G-1': 0.131, 'D-1': 0.955, 'M-5': 0.753,
}
lof_results = {
    'P-1': 0.706, 'S-1': 0.938, 'E-1': 0.806, 'E-2': 0.340,
    'F-1': 0.707, 'G-1': 0.272, 'D-1': 0.945, 'M-5': 0.182,
}
ocsvm_results = {
    'P-1': 0.678, 'S-1': 0.928, 'E-1': 0.862, 'E-2': 0.630,
    'F-1': 0.707, 'G-1': 0.093, 'D-1': 0.940, 'M-5': 0.360,
}

lstm_lof_f1 = {ch: results[ch]['f1'] for ch in channels}

comparison = pd.DataFrame([{
    'Channel': ch,
    'Anomaly': f"{results[ch]['anomaly_rate']:.1%}",
    'IF': f"{if_results.get(ch, 0):.1%}",
    'LSTM-VAE+IF': f"{deep_results.get(ch, 0):.1%}",
    'LOF': f"{lof_results.get(ch, 0):.1%}",
    'OCSVM': f"{ocsvm_results.get(ch, 0):.1%}",
    'LSTM-LOF+IF': f"{lstm_lof_f1.get(ch, 0):.1%}",
    'Best': max([('IF', if_results.get(ch, 0)),
                 ('LSTM-VAE+IF', deep_results.get(ch, 0)),
                 ('LOF', lof_results.get(ch, 0)),
                 ('OCSVM', ocsvm_results.get(ch, 0)),
                 ('LSTM-LOF+IF', lstm_lof_f1.get(ch, 0))],
                key=lambda x: x[1])[0],
} for ch in channels])

comparison

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
x = np.arange(len(channels))
w = 0.15

if_f1_vals = [if_results.get(ch, 0) for ch in channels]
deep_f1_vals = [deep_results.get(ch, 0) for ch in channels]
lof_f1_vals = [lof_results.get(ch, 0) for ch in channels]
ocsvm_f1_vals = [ocsvm_results.get(ch, 0) for ch in channels]
lstm_lof_f1_vals = [lstm_lof_f1.get(ch, 0) for ch in channels]

ax.bar(x - 2*w, if_f1_vals, w, label='IF', color='coral')
ax.bar(x - w, deep_f1_vals, w, label='LSTM-VAE+IF', color='mediumpurple')
ax.bar(x, lof_f1_vals, w, label='LOF', color='steelblue')
ax.bar(x + w, ocsvm_f1_vals, w, label='OCSVM', color='goldenrod')
ax.bar(x + 2*w, lstm_lof_f1_vals, w, label='LSTM-LOF+IF', color='seagreen')

ax.set_xticks(x)
ax.set_xticklabels(channels)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Channel F1: All Methods Comparison')
ax.legend(ncol=5, fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
macro_if = np.mean(list(if_results.values()))
macro_deep = np.mean(list(deep_results.values()))
macro_lof = np.mean(list(lof_results.values()))
macro_ocsvm = np.mean(list(ocsvm_results.values()))
macro_lstm_lof = macro_f1

fig, ax = plt.subplots(figsize=(10, 4))
methods = ['IF', 'LSTM-VAE+IF', 'LOF', 'OCSVM', 'LSTM-LOF+IF']
macro_f1s = [macro_if, macro_deep, macro_lof, macro_ocsvm, macro_lstm_lof]
colors = ['coral', 'mediumpurple', 'steelblue', 'goldenrod', 'seagreen']

bars = ax.bar(methods, macro_f1s, color=colors, width=0.55)
for bar, val in zip(bars, macro_f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.1%}', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('Macro F1 Score')
ax.set_title('Macro F1 Comparison Across All Methods (LSTM-LOF+IF = Best!)')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 0.85)

best_idx = macro_f1s.index(max(macro_f1s))
bars[best_idx].set_edgecolor('black')
bars[best_idx].set_linewidth(2)

plt.tight_layout()
plt.show()

---

## 15. Save Report

In [ ]:
summary = {
    'method': 'LSTM-Forecast + LOF + IF Ensemble',
    'micro_f1': round(micro_f1, 4),
    'macro_f1': round(macro_f1, 4),
    'micro_p': round(micro_p, 4),
    'micro_r': round(micro_r, 4),
    'per_channel': {ch: {
        'f1': round(results[ch]['f1'], 4),
        'best_method': results[ch]['best_method'],
        'LOF_err_f1': round(results[ch]['all']['LOF_err']['f1'], 4),
        'LOF_raw_f1': round(results[ch]['all']['LOF_raw']['f1'], 4),
        'IF_f1': round(results[ch]['all']['IF']['f1'], 4),
        'Ensemble_f1': round(results[ch]['all']['Ensemble']['f1'], 4),
    } for ch in channels},
}
os.makedirs('reports', exist_ok=True)
with open('reports/lstm_lof_metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved to reports/lstm_lof_metrics.json')